# Module 09 — AI Workbook Companion (GPU ray tracing)

A lightweight companion to
[RT_cores_ray_tracing_tutorial.ipynb](../RT_cores_ray_tracing_tutorial.ipynb). It
does **not** replace it — it adds the supervision layer from
[Module 11](../../11/README.md) to Module 09's topic: **ray–geometry
intersection**, the per-ray kernel that GPU RT cores accelerate.

New here? Read [Module 11's README](../../11/README.md) first. The failure mode
here is again one from the README list: an **"optimization" that changes the
math** — a plausible-looking intersection routine that returns the wrong surface.

## The loop, applied to ray–sphere intersection

Your task: for a batch of rays, find the nearest intersection distance `t` with a
sphere — the core of any ray tracer, and exactly the work an RT core does in
hardware. You may have an AI write it. Your job is everything around it.

1. **SPECIFY** — Contract: ray origins/directions (are directions normalised?),
   the sphere, what "correct" means (the **nearest** positive hit), and the metric.
2. **GENERATE** — Ask the AI for the vectorised intersection. Keep it separate.
3. **PREDICT** — *Before running:* the classic bug is taking the **far** root
   (`-b + √`) instead of the **near** root (`-b − √`), so you see the back surface
   of the sphere. Do you spot it?
4. **VERIFY** — Use [verify_raytrace.py](./verify_raytrace.py): a reference
   intersection with a per-ray comparison of `t`.
5. **DIAGNOSE** — Explain why the far-root version still renders a sphere-shaped
   blob (so a thumbnail looks fine) yet has wrong depths and normals.

## The adversarial exercise

[adversarial_raytrace_buggy.py](./adversarial_raytrace_buggy.py) is an
intersection routine "an AI wrote for you." It runs and reports hits for the
right pixels, but it returns the **far** intersection, so every depth is wrong —
which corrupts shading, shadows, and the normal at the hit point.

Your job:

1. Predict the bug from the choice of root *before* running.
2. Run it and compare the `t` values to the reference.
3. State the exact bug and the fix (choose the smallest positive root).
4. Prove the fix per-ray against the reference.

Could an AI catch this? Sometimes — but "it renders a sphere" passes a casual
thumbnail review while every distance is wrong. A per-ray `t` comparison catches
it. That is your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Why does taking the far root still draw a sphere-shaped region but with wrong
  depths? What visible artefacts would appear once you add shading or shadows?
- Why must the ray direction be normalised for `t` to be a real distance, and how
  would you test that assumption?
- On a GPU/RT core this runs per ray in parallel. What is the correctness check
  you would insist a generated kernel pass before trusting a rendered image?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Run the problem program [adversarial_raytrace_buggy.py](./adversarial_raytrace_buggy.py). Read it first and predict what will happen before you run this cell.

In [ ]:
!python adversarial_raytrace_buggy.py

## Step 2 - Run your verification harness

[verify_raytrace.py](./verify_raytrace.py) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the function under test. Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!python verify_raytrace.py

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.